# Momentum-flux divergence vs. vertical shear — TPOSE24 (3-hourly, from 2012-10-01; first 8 days spin-up dropped)

Is the vertical **momentum-flux divergence** $-\partial_z\langle u'w'\rangle,\,-\partial_z\langle v'w'\rangle$ (the wave force on the mean flow) *organised by* the **vertical shear** of the
flow? We test this on the **total** signal (anomaly from the window mean) and on
the **Yanai band** (15–40 day band-pass), against **zonal** $\partial_z U$,
**meridional** $\partial_z V$ and **total** $|S|=\sqrt{(\partial_zU)^2+(\partial_zV)^2}$ shear, using two shear variants:

- **background (mean) shear** $\partial_z\langle U\rangle$ — the flow the waves ride on;
- **band-matched fluctuating shear** $\partial_z u'$ (RMS profile / time series).

Correspondence is quantified two ways: **depth-structure** correlation (do the
vertical profiles line up?) and **temporal** correlation (do they co-vary in
time?). We report Pearson & Spearman $r$ with a *first-pass* significance
estimate — the definitive significance test is deliberately left open and will be
chosen after seeing these results. $r$ is the primary output.

In [1]:
import sys, os
sys.path.append('/home/edavenport/analysis/motive-yanai-waves/scripts')
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import wave_filter as wf, fluxes as fx, shear as sh, stats_corr as sc
import tpose24_io as io

CACHE   = io.CACHE_DIR
FIG_DIR = '/home/edavenport/analysis/motive-yanai-waves/figures'
COV3D   = f'{CACHE}/yanai_flux_cov3d_dt60.nc'
MEANUV  = f'{CACHE}/yanai_meanUV_maps_dt60.nc'
DT_DAYS = io.OUT_STEP_SEC / 86400.0
MAP_SUBSET = False  # cov3d/meanUV grid is a sub-domain slice?
ZMAX    = 1500.0            # depth range (m) for the correlations / zoom plots
# EDJ-focused depth bands (m, positive down): the equatorial deep jets live below
# the thermocline; the 0-300 band is kept only as a surface reference.
EDJ_BOUNDS = [(0, 300), (300, 700), (700, 1500), (1500, 2500), (2500, None)]
SEC_ZMAX = 3000.0           # depth limit (m) for the EDJ depth-sections
SEC_LON_DEG = 220.0         # 140W: longitude for the lat-depth section
os.makedirs(FIG_DIR, exist_ok=True)

## Load columns and define the two frequency treatments

Raw full-depth columns are loaded, normalised to `(time, lon, lat, depth)`, and
restricted to the analysis window (spin-up dropped where applicable, band-pass
edge-trim removed). The **total** perturbation is the anomaly from the window
mean; the **Yanai** perturbation is the 15–40 day band-pass.

In [2]:
prof = xr.open_dataset(f'{CACHE}/yanai_profiles_dt60.nc')
LONS    = ['140W']
LON_DEG = np.array([float(prof.attrs['lon'])])
LATS    = [str(x) for x in prof['loc'].values]      # 1S,0N,1N,2N,3N
LAT_DEG = prof['lat'].values
Z       = prof['depth'].values                      # cell-center depth (m,<0)
drF     = prof['drF'].values
hFac    = prof['hFacC'].values[None, :, :]          # (1,loc,depth)
# drop the first SPINUP_DAYS BEFORE filtering (never enters the band-pass)
time_all = prof['time'].values
cut = np.datetime64(io.REF_DATE) + np.timedelta64(int(io.SPINUP_DAYS*86400), 's')
spin = time_all > cut
times_pre = time_all[spin]
# normalise to (time, lon=1, lat=nloc, depth); center WVEL from faces (Zl)
U  = prof['UVEL'].values[spin][:, None, :, :]
V  = prof['VVEL'].values[spin][:, None, :, :]
Wc = fx.w_to_center(prof['WVEL'].values[spin][:, None, :, :], zaxis=-1)
RAW = {'U': U, 'V': V, 'Wc': Wc}
etm = io.edge_trim_mask(times_pre)                  # drop band-pass edge days

time = times_pre[etm]
nlon, nlat, nz = len(LONS), len(LATS), Z.size
print('lons =', dict(zip(LONS, np.round(LON_DEG, 2))),
      '| lats =', dict(zip(LATS, np.round(LAT_DEG, 2))))
print(f'analysis window: {str(time[0])[:10]} .. {str(time[-1])[:10]} '
      f'({time.size} steps, dt={DT_DAYS:.3f} d)')

# ---- frequency treatments -> perturbation velocities (time,lon,lat,depth) ----
Ut, Vt, Wt = RAW['U'][etm], RAW['V'][etm], RAW['Wc'][etm]
BANDS = {}
BANDS['total'] = {'u': Ut - np.nanmean(Ut, 0, keepdims=True),
                  'v': Vt - np.nanmean(Vt, 0, keepdims=True),
                  'w': Wt - np.nanmean(Wt, 0, keepdims=True)}
BANDS['yanai'] = {'u': wf.bandpass(RAW['U'], axis=0, dt_days=DT_DAYS)[etm],
                  'v': wf.bandpass(RAW['V'], axis=0, dt_days=DT_DAYS)[etm],
                  'w': wf.bandpass(RAW['Wc'], axis=0, dt_days=DT_DAYS)[etm]}

# ---- background (mean) shear, shared by both bands (lon,lat,depth) ----
Ubg, Vbg = np.nanmean(Ut, 0), np.nanmean(Vt, 0)
Sx0 = sh.vertical_shear(Ubg, Z, zaxis=-1)
Sy0 = sh.vertical_shear(Vbg, Z, zaxis=-1)
S0  = sh.shear_magnitude(Sx0, Sy0)
BG = {'Sx': Sx0, 'Sy': Sy0, 'S': S0}
print('bands:', list(BANDS), '| perturbation shape', BANDS['yanai']['u'].shape)

lons = {'140W': np.float64(220.02)} | lats = {'1S': np.float32(-0.98), '0N': np.float32(-0.02), '1N': np.float32(0.98), '2N': np.float32(2.02), '3N': np.float32(2.98)}
analysis window: 2012-10-29 .. 2012-12-09 (334 steps, dt=0.125 d)
bands: ['total', 'yanai'] | perturbation shape (334, 1, 5, 138)


## Divergence and band-matched shear for each band

For each band: the vertical momentum fluxes $u'w', v'w'$, their time-mean
divergence $-\partial_z\langle\cdot\rangle$ (and instantaneous divergence for
the temporal analysis), and the fluctuating shear $\partial_z u',\partial_z v'$
(instantaneous, plus its RMS profile).

In [3]:
DIV, SHR = {}, {}
for b, d in BANDS.items():
    uw, vw = d['u'] * d['w'], d['v'] * d['w']         # (time,lon,lat,depth)
    uw_m, vw_m = np.nanmean(uw, 0), np.nanmean(vw, 0)  # (lon,lat,depth)
    Dx = fx.vertical_convergence(uw_m, Z, zaxis=-1)    # -d<u'w'>/dz
    Dy = fx.vertical_convergence(vw_m, Z, zaxis=-1)
    Dmag = np.sqrt(Dx**2 + Dy**2)
    Dx_t = fx.vertical_convergence(uw, Z, zaxis=-1)    # instantaneous (time,...)
    Dy_t = fx.vertical_convergence(vw, Z, zaxis=-1)
    Dmag_t = np.sqrt(Dx_t**2 + Dy_t**2)
    DIV[b] = dict(Dx=Dx, Dy=Dy, Dmag=Dmag, Dx_t=Dx_t, Dy_t=Dy_t, Dmag_t=Dmag_t)
    Sx_t = sh.vertical_shear(d['u'], Z, zaxis=-1)      # fluctuating shear
    Sy_t = sh.vertical_shear(d['v'], Z, zaxis=-1)
    Sx_rms = np.sqrt(np.nanmean(Sx_t**2, 0))
    Sy_rms = np.sqrt(np.nanmean(Sy_t**2, 0))
    Srms = np.sqrt(Sx_rms**2 + Sy_rms**2)
    Smag_t = np.sqrt(Sx_t**2 + Sy_t**2)
    SHR[b] = dict(Sx_t=Sx_t, Sy_t=Sy_t, Smag_t=Smag_t,
                  Sx_rms=Sx_rms, Sy_rms=Sy_rms, Srms=Srms)
zsel = (-Z <= ZMAX)                # depth range used for the correlations
print('depth levels within', ZMAX, 'm:', int(zsel.sum()), 'of', nz)

depth levels within 1500.0 m: 98 of 138


## (d1) Divergence vs. background-shear profiles

Per longitude and band: blue = momentum-flux divergence (bottom axis), red =
background shear (top axis), overlaid vs depth so the vertical correspondence is
visible. Rows: zonal ($-\partial_z\langle u'w'\rangle$ vs $\partial_z U$),
meridional, total magnitude. The depth-structure Pearson $r$ over 0–ZMAX m is
printed in each panel.

In [4]:
def _sym(ax, prof, li, c, base0=False):
    d = prof[li, c][zsel]; d = d[np.isfinite(d)]
    m = np.nanmax(np.abs(d)) if d.size else 1.0
    m = m if m > 0 else 1.0
    ax.set_xlim(0, 1.05*m) if base0 else ax.set_xlim(-1.05*m, 1.05*m)

def d1_fig(li, LON, band):
    D = DIV[band]
    rows = [('zonal',  D['Dx'],   BG['Sx'], r"$-\partial_z\langle u'w'\rangle$", r"$\partial_z U$", False),
            ('merid.', D['Dy'],   BG['Sy'], r"$-\partial_z\langle v'w'\rangle$", r"$\partial_z V$", False),
            ('total',  D['Dmag'], BG['S'],  r"$|{-}\partial_z\langle u_iw\rangle|$", r"$|S_0|$", True)]
    fig, axes = plt.subplots(3, nlat, figsize=(3.1*nlat, 9.5), sharey=True,
                             squeeze=False)
    for r, (name, Dp, Sp, dl, sl, base0) in enumerate(rows):
        for c, L in enumerate(LATS):
            ax = axes[r, c]; axt = ax.twiny()
            ax.plot(Dp[li, c], Z, 'C0', lw=1.2)
            axt.plot(Sp[li, c], Z, 'C3', lw=1.1)
            if not base0: ax.axvline(0, color='0.6', lw=0.5)
            _sym(ax, Dp, li, c, base0); _sym(axt, Sp, li, c, base0)
            # axes auto-scaled per profile; suppress the (overlapping) numeric
            # ticks -- this panel compares SHAPE + r, blue=divergence, red=shear
            ax.tick_params(axis='x', which='both', labelbottom=False, length=0)
            axt.tick_params(axis='x', which='both', labeltop=False, length=0)
            for sp in ('bottom',): ax.spines[sp].set_color('C0')
            for sp in ('top',): axt.spines[sp].set_color('C3')
            res = sc.depth_corr(Dp[li, c][zsel], Sp[li, c][zsel], n_surro=0)
            ax.text(0.04, 0.03, f"r={res['r_pearson']:+.2f}", transform=ax.transAxes,
                    fontsize=8, va='bottom', ha='left',
                    bbox=dict(fc='white', ec='0.7', alpha=0.8, pad=1.2))
            if r == 0: axt.set_title(L, fontsize=9)
            if c == 0:
                ax.set_ylabel(name + '\ndepth (m)', fontsize=9)
            if r == 0 and c == nlat-1:
                ax.plot([], [], 'C0', label='divergence'); ax.plot([], [], 'C3', label='shear')
                ax.legend(fontsize=7, loc='lower right')
    axes[0, 0].set_ylim(-ZMAX, 0)
    fig.suptitle(f'{band} momentum-flux divergence (blue) vs background shear '
                 f'(red) at {LON}', y=0.995)
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/d1_shear_vs_divergence_{band}_{LON}.png', dpi=140)
    plt.close(fig)

for li, LON in enumerate(LONS):
    for band in ('total', 'yanai'):
        d1_fig(li, LON, band)
print('saved d1 profiles for', LONS)

saved d1 profiles for ['140W']


## (d2) Depth-structure correlation summary

Pearson $r$ across depth (0–ZMAX m) between the divergence and shear profiles,
for every column. Four heatmaps = {total, Yanai} band × {background, band-matched
(RMS)} shear; columns = zonal / meridional / total pairing. Background pairs use
signed profiles; band-matched pairs use magnitudes ($|D|$ vs RMS shear). `*`
marks a first-pass surrogate $p<0.05$ (provisional).

In [5]:
LOC_LABELS = [f'{LON},{L}' for LON in LONS for L in LATS]
PAIRS = ['zonal', 'merid', 'total']

def _profiles(band, variant):
    # return dict pairing -> (div_profile_fn, shear_profile) over (lon,lat,depth)
    D, S = DIV[band], SHR[band]
    if variant == 'background':
        return {'zonal': (D['Dx'], BG['Sx']), 'merid': (D['Dy'], BG['Sy']),
                'total': (D['Dmag'], BG['S'])}
    else:  # band-matched RMS shear vs divergence magnitude
        return {'zonal': (np.abs(D['Dx']), S['Sx_rms']),
                'merid': (np.abs(D['Dy']), S['Sy_rms']),
                'total': (D['Dmag'], S['Srms'])}

combos = [('total', 'background'), ('yanai', 'background'),
          ('total', 'band-matched'), ('yanai', 'band-matched')]

def d2_summary(zmin, zmax, label, fname):
    zmask = (-Z >= zmin) & (-Z <= zmax)
    fig, axes = plt.subplots(1, 4, figsize=(15, 0.42*len(LOC_LABELS)+2.2), squeeze=False)
    for a, (band, variant) in enumerate(combos):
        P = _profiles(band, variant)
        R = np.full((len(LOC_LABELS), 3), np.nan)
        Pv = np.full((len(LOC_LABELS), 3), np.nan)
        for j, pair in enumerate(PAIRS):
            Dp, Sp = P[pair]
            i = 0
            for li in range(nlon):
                for c in range(nlat):
                    res = sc.depth_corr(Dp[li, c][zmask], Sp[li, c][zmask], n_surro=800)
                    R[i, j] = res['r_pearson']; Pv[i, j] = res['p_surrogate']; i += 1
        ax = axes[0, a]
        im = ax.imshow(R, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
        ax.set_xticks(range(3)); ax.set_xticklabels(PAIRS, fontsize=8)
        ax.set_yticks(range(len(LOC_LABELS)))
        ax.set_yticklabels(LOC_LABELS if a == 0 else [], fontsize=7)
        ax.set_title(f'{band}\n{variant} shear', fontsize=9)
        for i in range(len(LOC_LABELS)):
            for j in range(3):
                if np.isfinite(R[i, j]):
                    star = '*' if (np.isfinite(Pv[i, j]) and Pv[i, j] < 0.05) else ''
                    ax.text(j, i, f'{R[i, j]:+.2f}{star}', ha='center', va='center',
                            fontsize=6, color='k')
    fig.colorbar(im, ax=axes[0, :].tolist(), shrink=0.7, pad=0.01, label='Pearson r')
    fig.suptitle(f'Depth-structure correlation of momentum-flux divergence vs shear '
                 f'({label})', y=1.02)
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=140, bbox_inches='tight')
    plt.close(fig)

d2_summary(0, ZMAX, f'0-{ZMAX:.0f} m', 'd2_depth_correlation_summary.png')
# EDJ-focused deep range (below the thermocline, where the deep jets live)
d2_summary(300, 2500, '300-2500 m (EDJ)', 'd2_depth_correlation_summary_EDJ.png')
print('saved d2 depth-correlation summaries (full + EDJ deep range)')

saved d2 depth-correlation summaries (full + EDJ deep range)


## (d3) Temporal correlation vs. depth (band-matched shear)

At each depth, Pearson $r$ between the instantaneous divergence and the
band-matched fluctuating shear time series, per latitude. Shaded where the
first-pass effective-DOF $p<0.05$ (provisional). **Caveat:** divergence and shear
both derive from $u'$, so some correlation is structural.

In [6]:
def d3_fig(li, LON, band):
    D, S = DIV[band], SHR[band]
    rows = [('zonal',  D['Dx_t'],   S['Sx_t']),
            ('merid.', D['Dy_t'],   S['Sy_t']),
            ('total',  D['Dmag_t'], S['Smag_t'])]
    fig, axes = plt.subplots(3, nlat, figsize=(3.0*nlat, 9), sharey=True,
                             sharex=True, squeeze=False)
    for r, (name, A, B) in enumerate(rows):
        for c, L in enumerate(LATS):
            ax = axes[r, c]
            res = sc.temporal_corr_profile(A[:, li, c, :], B[:, li, c, :])
            rr, pp = res['r'], res['p']
            ax.plot(rr, Z, 'C4', lw=1.2)
            sig = np.isfinite(pp) & (pp < 0.05)
            ax.fill_betweenx(Z, 0, np.where(sig, rr, 0), color='C4', alpha=0.25)
            ax.axvline(0, color='0.6', lw=0.5); ax.set_xlim(-1, 1)
            ax.tick_params(labelsize=7)
            if r == 0: ax.set_title(L, fontsize=9)
            if c == 0: ax.set_ylabel(name + ' r\ndepth (m)', fontsize=9)
    axes[0, 0].set_ylim(-ZMAX, 0)
    fig.suptitle(f'{band} temporal correlation: divergence vs band-matched shear '
                 f'at {LON} (shaded p<0.05, provisional)', y=0.996)
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/d3_temporal_correlation_{band}_{LON}.png', dpi=140)
    plt.close(fig)

for li, LON in enumerate(LONS):
    for band in ('total', 'yanai'):
        d3_fig(li, LON, band)
print('saved d3 temporal-correlation profiles for', LONS)

saved d3 temporal-correlation profiles for ['140W']


## (d4) Depth-structure correlation maps (Yanai band, background shear)

One $r$ per column: depth-structure Pearson correlation (0–ZMAX m) between the
Yanai-band mean-flux divergence (from the 3-D covariance cache) and the
background shear (from the time-mean U/V cache). Black crosses mark the profile
columns. Requires `COV3D` and `MEANUV` caches.

In [7]:
# Load the 3-D Yanai-band covariances + background mean-flow caches ONCE; the d4
# maps and the EDJ maps/sections below all reuse these globals.
MAPS_OK = True
try:
    cov = xr.open_dataset(COV3D); mv = xr.open_dataset(MEANUV)
    Zc  = cov['depth'].values                         # (depth,) cell-center m,<0
    drFc = cov['drF'].values
    lon2, lat2 = cov['lon'].values, cov['lat'].values
    uw_m = cov['cov'].sel(component='uw').values       # (depth,y,x) time-mean flux
    vw_m = cov['cov'].sel(component='vw').values
    Fmag = np.sqrt(uw_m**2 + vw_m**2)                  # vertical mom-flux magnitude
    Dx0 = fx.vertical_convergence(uw_m, Zc, zaxis=0)   # -dz<u'w'>  (zonal force)
    Dy0 = fx.vertical_convergence(vw_m, Zc, zaxis=0)
    Dm0 = np.sqrt(Dx0**2 + Dy0**2)                     # flux-divergence magnitude
    Umean, Vmean = mv['Umean'].values, mv['Vmean'].values
    Sx0m, Sy0m, S0m = mv['Sx0'].values, mv['Sy0'].values, mv['S0'].values
    # partial-cell fraction on the map grid (for depth-weighted layer averages)
    _g = io.load_grid()
    if MAP_SUBSET:
        iy, ix = io.map_subset_idx(_g)
        hf_map = _g['hFacC'][:, iy.min():iy.max()+1, ix.min():ix.max()+1]
    else:
        hf_map = _g['hFacC']
    mk_lon = np.repeat(LON_DEG, len(LAT_DEG)); mk_lat = np.tile(LAT_DEG, len(LON_DEG))
    print('maps grid', uw_m.shape, '| hFacC', hf_map.shape)
except FileNotFoundError as e:
    MAPS_OK = False
    print('SKIP maps/sections -- missing cache:', e)

maps grid (138, 384, 512) | hFacC (138, 384, 512)


In [8]:
def rmap(A, B, zmask):
    a, b = A[zmask], B[zmask]               # (nz_sel, y, x)
    m = np.isfinite(a) & np.isfinite(b)
    a = np.where(m, a, np.nan); b = np.where(m, b, np.nan)
    ap = a - np.nanmean(a, 0); bp = b - np.nanmean(b, 0)
    num = np.nansum(ap*bp, 0)
    den = np.sqrt(np.nansum(ap**2, 0) * np.nansum(bp**2, 0))
    with np.errstate(invalid='ignore', divide='ignore'):
        r = num / den
    r[(den == 0) | (np.sum(m, 0) < 5)] = np.nan
    return r

def d4_maps(zmin, zmax, label, fname):
    zmask = (-Zc >= zmin) & (-Zc <= zmax)
    panels = [('zonal  (Dx vs \u2202zU)', rmap(Dx0, Sx0m, zmask)),
              ('merid. (Dy vs \u2202zV)', rmap(Dy0, Sy0m, zmask)),
              ('total  (|D| vs |S0|)',    rmap(Dm0, S0m, zmask))]
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), sharey=True, squeeze=False,
                             layout='constrained')
    for a, (name, R) in enumerate(panels):
        ax = axes[0, a]
        im = ax.pcolormesh(lon2, lat2, R, cmap='RdBu_r', vmin=-1, vmax=1, shading='auto')
        ax.plot(mk_lon, mk_lat, 'kx', ms=4)
        ax.axhline(0, color='k', lw=0.3); ax.set_title(name, fontsize=10)
        ax.set_xlabel('lon (\u00b0E)')
        if a == 0: ax.set_ylabel('lat (\u00b0N)')
    fig.colorbar(im, ax=axes[0, :].tolist(), shrink=0.85, label='depth-structure r')
    fig.suptitle(f'Yanai-band momentum-flux divergence vs background shear: '
                 f'depth-structure correlation ({label})')
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=140, bbox_inches='tight')
    plt.close(fig)

if MAPS_OK:
    d4_maps(0, ZMAX, f'0-{ZMAX:.0f} m', 'd4_correlation_maps.png')
    d4_maps(300, 2500, '300-2500 m (EDJ)', 'd4_correlation_maps_EDJ.png')
    print('saved d4 correlation maps (full + EDJ deep range)')

saved d4 correlation maps (full + EDJ deep range)


## (e1) How big are the terms? — depth-averaged magnitude maps by EDJ layer

Partial-cell-weighted **depth-average** within each EDJ-focused band of three
magnitudes: the **Yanai vertical momentum flux** $|\langle\mathbf{u}'w'\rangle|$, its **divergence** $|{-}\partial_z\langle\mathbf{u}'w'\rangle|$, and the **background shear**
$|S_0|$. Reading across a column shows where a large flux does or does not produce
large divergence, and whether that tracks the background shear. The 0–300 m band
is a surface reference; the deeper bands are where the equatorial deep jets live.

In [9]:
def _lmask(top, bot):
    d = -Zc; hi = np.inf if bot is None else bot
    return (d >= top) & (d < hi)

if MAPS_OK:
    LAYER_LABELS = [f'{t}-{b}m' if b is not None else f'{t}m-bottom'
                    for t, b in EDJ_BOUNDS]
    terms = [(r"$|\langle \mathbf{u}'w'\rangle|$ (m$^2$s$^{-2}$)", Fmag),
             (r"$|{-}\partial_z\langle \mathbf{u}'w'\rangle|$ (m s$^{-2}$)", Dm0),
             (r"$|S_0|$ (s$^{-1}$)", S0m)]
    nc = len(EDJ_BOUNDS)
    fig, axes = plt.subplots(3, nc, figsize=(3.0*nc, 8.4), sharex=True, sharey=True,
                             squeeze=False, layout='constrained')
    for r, (lab, F) in enumerate(terms):
        avgs = [fx.layer_average(F, drFc, hf_map, _lmask(t, b)) for t, b in EDJ_BOUNDS]
        # scale colors to the DEEP bands (exclude the 0-300 m surface reference,
        # which is orders of magnitude larger and would wash out the deep jets)
        deep = np.abs(np.stack(avgs[1:]))
        v98 = np.nanpercentile(deep, 98) if np.isfinite(deep).any() else 1.0
        v98 = v98 if v98 > 0 else 1.0
        for c, A in enumerate(avgs):
            ax = axes[r, c]
            im = ax.pcolormesh(lon2, lat2, A, cmap='viridis', vmin=0, vmax=v98,
                               shading='auto')
            ax.plot(mk_lon, mk_lat, 'kx', ms=3); ax.axhline(0, color='w', lw=0.3)
            if r == 0:
                ttl = LAYER_LABELS[c] + (' (ref, saturated)' if c == 0 else '')
                ax.set_title(ttl, fontsize=9)
            if r == 2: ax.set_xlabel('lon (\u00b0E)', fontsize=8)
            if c == 0: ax.set_ylabel(lab + '\nlat (\u00b0N)', fontsize=8)
        fig.colorbar(im, ax=axes[r, :].tolist(), shrink=0.8, pad=0.01)
    fig.suptitle('Depth-averaged magnitudes by EDJ layer: Yanai flux, its '
                 'divergence, and background shear')
    fig.savefig(f'{FIG_DIR}/e1_magnitude_maps_by_layer.png', dpi=140, bbox_inches='tight')
    plt.close(fig)
    print('saved e1 magnitude maps by EDJ layer')

saved e1 magnitude maps by EDJ layer


## (e2, e3) EDJ depth sections — do the wave forcing and shear line up with the jets?

Depth sections of the **background zonal velocity** $\langle U\rangle$ (the
stacked deep jets), the **Yanai zonal flux** $\langle u'w'\rangle$, the **Yanai
zonal force** $-\partial_z\langle u'w'\rangle$, and the **background zonal shear**
$\partial_z\langle U\rangle$. e2 is a lon–depth section at the equator; e3 is a
lat–depth section at 140°W. Signed (diverging) colormaps, **scaled to the
sub-thermocline (>250 m) signal** so the deep jets are visible (the near-surface
EUC/thermocline saturates and is not the focus).

In [10]:
def _section(fields, xcoord, xlabel, title, fname, xslice=None):
    zmask = -Zc <= SEC_ZMAX
    zc = Zc[zmask]
    fig, axes = plt.subplots(len(fields), 1, figsize=(9, 2.4*len(fields)+0.5),
                             sharex=True, squeeze=False)
    x = xcoord if xslice is None else xcoord[xslice]
    for r, (lab, F, unit) in enumerate(fields):
        S = F[zmask]                                   # (nz_sel, n)
        if xslice is not None: S = S[:, xslice]
        # scale to the sub-thermocline (>250 m) signal so the deep jets are
        # visible; the near-surface EUC/thermocline saturates (not the focus)
        deep = S[zc < -250]
        dfin = np.abs(deep[np.isfinite(deep)])
        v98 = np.nanpercentile(dfin, 98) if dfin.size else np.nanpercentile(np.abs(S), 98)
        v98 = v98 if v98 > 0 else 1.0
        ax = axes[r, 0]
        m = ax.pcolormesh(x, zc, S, cmap='RdBu_r', vmin=-v98, vmax=v98, shading='auto')
        ax.set_ylabel(lab + '\ndepth (m)', fontsize=9)
        cb = fig.colorbar(m, ax=ax, shrink=0.9, pad=0.01); cb.set_label(unit, fontsize=8)
        cb.formatter.set_powerlimits((-2, 2)); cb.update_ticks()
    axes[-1, 0].set_xlabel(xlabel)
    fig.suptitle(title, y=0.999)
    fig.tight_layout()
    fig.savefig(f'{FIG_DIR}/{fname}', dpi=140, bbox_inches='tight')
    plt.close(fig)

if MAPS_OK:
    zon = [(r"$\langle U\rangle$", Umean, 'm s$^{-1}$'),
           (r"$\langle u'w'\rangle$", uw_m, 'm$^2$ s$^{-2}$'),
           (r"$-\partial_z\langle u'w'\rangle$", Dx0, 'm s$^{-2}$'),
           (r"$\partial_z\langle U\rangle$", Sx0m, 's$^{-1}$')]
    # e2: lon-depth section at the equator (nearest latitude to 0)
    jeq = int(np.nanargmin(np.abs(lat2)))
    fields_eq = [(lab, F[:, jeq, :], unit) for lab, F, unit in zon]
    _section(fields_eq, lon2, 'lon (\u00b0E)',
             f'EDJ section at the equator (lat={lat2[jeq]:.2f}\u00b0): background '
             f'jets, Yanai zonal flux/force, background shear',
             'e2_section_lon_depth_equator.png')
    # e3: lat-depth section at 140W
    i140 = int(np.nanargmin(np.abs(lon2 - SEC_LON_DEG)))
    fields_140 = [(lab, F[:, :, i140], unit) for lab, F, unit in zon]
    latsel = (lat2 >= -5) & (lat2 <= 8)
    _section(fields_140, lat2, 'lat (\u00b0N)',
             f'EDJ section at 140\u00b0W (lon={lon2[i140]:.1f}\u00b0E): background '
             f'jets, Yanai zonal flux/force, background shear',
             'e3_section_lat_depth_140W.png', xslice=latsel)
    print('saved e2 (equator) and e3 (140W) EDJ depth sections')

saved e2 (equator) and e3 (140W) EDJ depth sections
